In [1]:
import os
import json
import base64
import requests

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# ========== 环境参数（优先从 .env 读取，否则使用默认值） ==========
# nginx 版本只需要 HOST_IP + NGINX_PORT，所有服务通过路由区分
HOST_IP = os.getenv("HOST_IP", "192.168.1.13")
NGINX_PORT = int(os.getenv("NGINX_PORT", "10000"))

API_KEY = os.getenv("API_KEY", "SII#gemr#2026!!")

# nginx 网关基础地址
BASE_URL = f"http://{HOST_IP}:{NGINX_PORT}"


def auth_headers() -> dict:
    """构造带 Bearer Token 的请求头。"""
    return {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }


def post_json(url: str, payload: dict, timeout: int = 300) -> dict:
    """发送 POST 请求并返回 JSON 响应（非流式）。"""
    resp = requests.post(url, headers=auth_headers(), json=payload, timeout=timeout)
    resp.raise_for_status()
    return resp.json()


def iter_sse_lines(response):
    """从流式 HTTP 响应中逐行解析 SSE data 字段。"""
    for raw in response.iter_lines(decode_unicode=True):
        if not raw:
            continue
        if raw.startswith("data: "):
            yield raw[len("data: "):].strip()

In [ ]:
# ========== LLM 文本对话（requests via nginx） ==========

url = f"{BASE_URL}/llm/v1/chat/completions"

payload = {
    "model": "LLM",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "解释什么是function call"},
    ],
    "temperature": 0.2,
}

# --- 非流式 ---
print("=== LLM 非流式 ===")
data = post_json(url, payload)
print(data["choices"][0]["message"]["content"])

# --- 流式 ---
print("\n=== LLM 流式 ===")
payload_stream = {**payload, "stream": True}
with requests.post(url, headers=auth_headers(), json=payload_stream,
                   stream=True, timeout=300) as response:
    response.raise_for_status()
    for sse in iter_sse_lines(response):
        if sse == "[DONE]":
            break
        event_data = json.loads(sse)
        # 流式返回的增量内容在 delta.content 中
        content = event_data.get("choices", [{}])[0].get("delta", {}).get("content")
        if content:
            print(content, end="", flush=True)
print()

In [ ]:
# ========== VLM 图文理解（requests via nginx） ==========

local_image = "未知2_1769051564.jpg"  # 请确保文件存在
if not os.path.exists(local_image):
    raise FileNotFoundError(f"image file not found: {local_image}")

# 读取图片并 base64 编码
with open(local_image, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

url = f"{BASE_URL}/vlm/v1/chat/completions"

payload = {
    "model": "VLM",
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "请描述图片里最主要的内容。"},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
                },
            ],
        }
    ],
    "temperature": 0.2,
}

# --- 非流式 ---
print("=== VLM 非流式 ===")
data = post_json(url, payload)
print(data["choices"][0]["message"]["content"])

# --- 流式 ---
print("\n=== VLM 流式 ===")
payload_stream = {**payload, "stream": True}
with requests.post(url, headers=auth_headers(), json=payload_stream,
                   stream=True, timeout=300) as response:
    response.raise_for_status()
    for sse in iter_sse_lines(response):
        if sse == "[DONE]":
            break
        event_data = json.loads(sse)
        # 流式返回的增量内容在 delta.content 中
        content = event_data.get("choices", [{}])[0].get("delta", {}).get("content")
        if content:
            print(content, end="", flush=True)
print()

In [5]:
# ========== ASR 语音识别（requests via nginx） ==========

local_audio = "audio-1.wav"  # 请确保文件存在
if not os.path.exists(local_audio):
    raise FileNotFoundError(f"audio file not found: {local_audio}")

# 读取音频并 base64 编码
with open(local_audio, "rb") as f:
    audio_b64 = base64.b64encode(f.read()).decode("utf-8")

url = f"{BASE_URL}/asr/v1/chat/completions"

payload = {
    "model": "ASR",
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "audio_url",
                    "audio_url": {"url": f"data:audio/wav;base64,{audio_b64}"},
                }
            ],
        }
    ],
}

# --- 非流式 ---
print("=== ASR 非流式 ===")
data = post_json(url, payload)
print(data["choices"][0]["message"]["content"])

# --- 流式 ---
print("\n=== ASR 流式 ===")
payload_stream = {**payload, "stream": True}
with requests.post(url, headers=auth_headers(), json=payload_stream,
                   stream=True, timeout=300) as response:
    response.raise_for_status()
    for sse in iter_sse_lines(response):
        if sse == "[DONE]":
            break
        event_data = json.loads(sse)
        content = event_data.get("choices", [{}])[0].get("delta", {}).get("content")
        if content:
            print(content, end="", flush=True)
print()

=== ASR 非流式 ===
language Chinese<asr_text>选择有价值的应用场景，然后研究如何通过模型能力实现功能。

=== ASR 流式 ===
language Chinese<asr_text>选择有价值的应用场景，然后研究如何通过模型能力实现功能。


In [ ]:
# ========== Embedding 向量化（requests via nginx） ==========

url = f"{BASE_URL}/embedding/v1/embeddings"

payload = {
    "model": "EMBEDDING",
    "input": [
        "向量数据库适合存什么？",
        "embedding 有什么用？",
    ],
}

data = post_json(url, payload)
# 打印第一条文本的向量维度和前 20 个分量
print("dim =", len(data["data"][0]["embedding"]))
print("first 20 =", data["data"][0]["embedding"][:20])

In [ ]:
# ========== Reranker 重排序（requests via nginx） ==========
# 注意：reranker 服务使用 /rerank 端点，nginx 路由前缀也是 /rerank

url = f"{BASE_URL}/rerank/rerank"

payload = {
    "model": "RERANKER",
    "query": "我是一只猫？",
    "documents": [
        "我是一只狗",
        "我也是一只猫",
        "你可以在 APP 里提交申请。",
        "退税需要满足一定条件，并准备相关材料。",
        "今天天气不错。",
    ],
    "top_n": 5,
}

try:
    data = post_json(url, payload)
    print(data)
except Exception as e:
    print("reranker 调用失败:", e)